In [1]:
!pip install requests beautifulsoup4 pandas transformers torch scikit-learn plotly tqdm

In [4]:
import requests
from bs4 import BeautifulSoup
import re
import time
import pandas as pd
import numpy as np
import torch

from urllib.parse import urljoin, urlparse
from tqdm.auto import tqdm

from transformers import AutoTokenizer, AutoModel
from sklearn.metrics.pairwise import cosine_similarity


def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)
    text = text.strip()
    return text


def get_article_links(initial_url, base_url, headers, max_pages=2):
    all_article_links = set()
    visited_pagination_links = set()
    queue = [(initial_url, 1)]

    print("--- Iniciando coleta de links de artigos ---")

    while queue:
        current_url, page_number = queue.pop(0)

        if current_url in visited_pagination_links or page_number > max_pages:
            continue

        visited_pagination_links.add(current_url)
        print(f"Visitando listagem: Página {page_number} -> {current_url}")

        try:
            response = requests.get(current_url, headers=headers, timeout=15)
            response.raise_for_status()

            soup = BeautifulSoup(response.content, 'html.parser')

            for link_tag in soup.find_all('a', href=True):
                href = link_tag['href']
                full_url = urljoin(base_url, href)

                parsed = urlparse(full_url)
                path = parsed.path

                if '/doencas-e-sintomas/' not in path:
                    continue

                if path == '/doencas-e-sintomas/':
                    continue

                if '/page/' in path:
                    if full_url not in visited_pagination_links and page_number < max_pages:
                        queue.append((full_url, page_number + 1))
                else:
                    all_article_links.add(full_url)

            time.sleep(1)

        except Exception as e:
            print(f"Erro ao acessar {current_url}: {e}")

    return sorted(list(all_article_links))


def scrape_symptoms_only(article_url, headers):
    try:
        response = requests.get(article_url, headers=headers, timeout=15)
        response.raise_for_status()

        soup = BeautifulSoup(response.content, 'html.parser')

        main_content = soup.find('main')

        if main_content is None:
            main_content = soup.find('article')

        if main_content is None:
            return None

        title_tag = main_content.find('h1')
        disease_name = title_tag.get_text(strip=True) if title_tag else "Não encontrado"

        symptoms_paragraphs = []
        symptom_section_active = False

        elements = main_content.find_all(['h2', 'h3', 'p'])

        for el in elements:
            text = el.get_text(strip=True)

            if not text:
                continue

            if el.name in ['h2', 'h3'] and 'sintoma' in text.lower():
                symptom_section_active = True
                continue

            if symptom_section_active:
                if el.name == 'p':
                    symptoms_paragraphs.append(text)
                elif el.name in ['h2', 'h3']:
                    break

        full_symptoms = " ".join(symptoms_paragraphs)

        if full_symptoms:
            symptoms_text = preprocess_text(full_symptoms)
        else:
            symptoms_text = "Sintomas não detalhados nesta seção"

        return {
            'link': article_url,
            'nome_doenca': disease_name,
            'sintomas': symptoms_text
        }

    except Exception as e:
        print(f"Erro ao processar {article_url}: {e}")
        return None


def get_device():
    return "cuda" if torch.cuda.is_available() else "cpu"


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts


@torch.no_grad()
def gerar_embeddings_bert(textos, model_name="neuralmind/bert-base-portuguese-cased", batch_size=8):
    device = get_device()

    print(f"\nUsando dispositivo: {device}")
    print(f"Carregando modelo: {model_name}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name).to(device)

    model.eval()

    all_embeddings = []

    for i in tqdm(range(0, len(textos), batch_size), desc="Gerando embeddings com BERT"):
        batch = textos[i:i + batch_size]

        inputs = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=512,
            return_tensors="pt"
        )

        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"]
        )

        embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu().numpy())

    return np.vstack(all_embeddings)


def buscar_doencas_semelhantes(nome_doenca, nomes_doencas, textos_sintomas, matriz_similaridade, top_n=5):
    if nome_doenca not in nomes_doencas:
        print("Doença não encontrada.")
        print("\nDoenças disponíveis:")
        for nome in nomes_doencas:
            print("-", nome)
        return None

    idx = nomes_doencas.index(nome_doenca)

    scores = list(enumerate(matriz_similaridade[idx]))
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    resultados = []

    for i, score in scores[1:top_n + 1]:
        resultados.append({
            "doenca": nomes_doencas[i],
            "similaridade": round(float(score), 4),
            "sintomas": textos_sintomas[i]
        })

    return pd.DataFrame(resultados)


BASE_URL = "https://drauziovarella.uol.com.br"
START_URL = f"{BASE_URL}/doencas-e-sintomas/"

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

MAX_PAGES = 2
MAX_ARTIGOS = 30

MODELO_BERT = "neuralmind/bert-base-portuguese-cased"

links = get_article_links(
    initial_url=START_URL,
    base_url=BASE_URL,
    headers=HEADERS,
    max_pages=MAX_PAGES
)

print(f"\nTotal de links encontrados: {len(links)}")
print(f"Serão processados até {MAX_ARTIGOS} artigos.")

resultados = []

for i, url in enumerate(tqdm(links[:MAX_ARTIGOS], desc="Extraindo sintomas")):
    print(f"[{i+1}/{min(MAX_ARTIGOS, len(links))}] Extraindo sintomas de: {url}")

    dados = scrape_symptoms_only(url, HEADERS)

    if dados:
        resultados.append(dados)

    time.sleep(1)

df = pd.DataFrame(resultados)

df.to_csv('sintomas_doencas.csv', index=False, encoding='utf-8-sig')

print("\n--- Coleta concluída ---")
print("Arquivo 'sintomas_doencas.csv' gerado com sucesso!")

display(df.head())

df = df.dropna(subset=['sintomas'])
df = df[df['sintomas'].str.strip() != ""]
df = df[df['sintomas'] != "Sintomas não detalhados nesta seção"]

df = df.reset_index(drop=True)

print(f"\nTotal de doenças com sintomas válidos: {len(df)}")

display(df[['nome_doenca', 'sintomas']].head())

if len(df) == 0:
    print("Nenhuma doença com sintomas válidos foi encontrada. Não será possível aplicar BERT.")

else:
    textos_sintomas = df['sintomas'].tolist()
    nomes_doencas = df['nome_doenca'].tolist()

    X = gerar_embeddings_bert(
        textos=textos_sintomas,
        model_name=MODELO_BERT,
        batch_size=8
    )

    print("\nFormato da matriz de embeddings:")
    print(X.shape)

    print("\nInterpretação:")
    print(f"Foram gerados embeddings para {X.shape[0]} doenças.")
    print(f"Cada doença foi representada por um vetor com {X.shape[1]} números.")

    matriz_similaridade = cosine_similarity(X)

    similaridade_df = pd.DataFrame(
        matriz_similaridade,
        index=nomes_doencas,
        columns=nomes_doencas
    )

    print("\nMatriz de similaridade entre doenças:")
    display(similaridade_df)

    doenca_exemplo = nomes_doencas[0]

    print("\nDoença usada como exemplo:")
    print(doenca_exemplo)

    print("\nDoenças com sintomas mais semelhantes:")

    resultado_busca = buscar_doencas_semelhantes(
        nome_doenca=doenca_exemplo,
        nomes_doencas=nomes_doencas,
        textos_sintomas=textos_sintomas,
        matriz_similaridade=matriz_similaridade,
        top_n=5
    )

    display(resultado_busca)

    colunas_embeddings = [f"embedding_{i}" for i in range(X.shape[1])]

    df_embeddings = pd.DataFrame(X, columns=colunas_embeddings)

    df_final = pd.concat(
        [
            df[['link', 'nome_doenca', 'sintomas']],
            df_embeddings
        ],
        axis=1
    )

    df_final.to_csv(
        'doencas_sintomas_embeddings_bert.csv',
        index=False,
        encoding='utf-8-sig'
    )

    similaridade_df.to_csv(
        'matriz_similaridade_doencas_bert.csv',
        encoding='utf-8-sig'
    )

    print("\nArquivos finais gerados:")
    print("- sintomas_doencas.csv")
    print("- doencas_sintomas_embeddings_bert.csv")
    print("- matriz_similaridade_doencas_bert.csv")

--- Iniciando coleta de links de artigos ---
Visitando listagem: Página 1 -> https://drauziovarella.uol.com.br/doencas-e-sintomas/
Visitando listagem: Página 2 -> https://drauziovarella.uol.com.br/doencas-e-sintomas/page/2/

Total de links encontrados: 15
Serão processados até 30 artigos.


Extraindo sintomas:   0%|          | 0/15 [00:00<?, ?it/s]

[1/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/anemia-aplastica/
[2/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/anemia-perniciosa-o-que-e-sintomas-e-tratamento/
[3/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/cancer-de-baco-veja-como-identificar-e-tratar-essa-doenca-rara/
[4/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/disfuncao-da-articulacao-temporomandibular-atm-o-que-e-e-como-tratar/
[5/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/disfuncao-eretil-conheca-os-sintomas-causas-e-tratamento/
[6/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/doenca-de-lyme/
[7/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/fratura-de-costela/
[8/15] Extraindo sintomas de: https://drauziovarella.uol.com.br/doencas-e-sintomas/gastrite-atrofica-o-que-e-e-quais-os-sint

,link,nome_doenca,sintomas
0,https://drauziovarella.uol.com.br/doencas-e-si...,Anemia aplástica,os sintomas dependem das células sanguíneas at...
1,https://drauziovarella.uol.com.br/doencas-e-si...,"Anemia perniciosa: o que é, sintomas e tratamento",os sintomas mais comuns são alteração no funci...
2,https://drauziovarella.uol.com.br/doencas-e-si...,Câncer de baço: veja como identificar e tratar...,osprincipais sintomas do câncer de baçosão: ve...
3,https://drauziovarella.uol.com.br/doencas-e-si...,Disfunção da articulação temporomandibular (AT...,os principais sintomas da disfunção da articul...
4,https://drauziovarella.uol.com.br/doencas-e-si...,"Disfunção erétil: conheça os sintomas, causas ...",devido à sua característica principal — não co...



Total de doenças com sintomas válidos: 9


...


Formato da matriz de embeddings:
(9, 768)

Interpretação:
Foram gerados embeddings para 9 doenças.
Cada doença foi representada por um vetor com 768 números.

Matriz de similaridade entre doenças:


,Anemia aplástica,"Anemia perniciosa: o que é, sintomas e tratamento",Câncer de baço: veja como identificar e tratar essa doença rara,Disfunção da articulação temporomandibular (ATM): o que é e como tratar?,"Disfunção erétil: conheça os sintomas, causas e tratamento",Fratura de costela,Gastrite atrófica: o que é e quais os sintomas,Hematúria (presença de sangue na urina),Obesidade
Anemia aplástica,1.000000,0.896233,0.714391,0.622217,0.776812,0.843959,0.753141,0.858827,0.817472
"Anemia perniciosa: o que é, sintomas e tratamento",0.896233,1.000000,0.712988,0.680067,0.849258,0.876608,0.735729,0.901544,0.866749
Câncer de baço: veja como identificar e tratar essa doença rara,0.714391,0.712988,1.000000,0.590058,0.714596,0.740951,0.767473,0.733970,0.719814
Disfunção da articulação temporomandibular (ATM): o que é e como tratar?,0.622217,0.680067,0.590058,1.000000,0.636510,0.650035,0.650436,0.600916,0.675203
"Disfunção erétil: conheça os sintomas, causas e tratamento",0.776812,0.849258,0.714596,0.636510,1.000000,0.836216,0.695966,0.840756,0.823171
Fratura de costela,0.843959,0.876608,0.740951,0.650035,0.836216,1.000000,0.743383,0.898230,0.834860
Gastrite atrófica: o que é e quais os sintomas,0.753141,0.735729,0.767473,0.650436,0.695966,0.743383,1.000000,0.725116,0.762545
Hematúria (presença de sangue na urina),0.858827,0.901544,0.733970,0.600916,0.840756,0.898230,0.725116,1.000000,0.854560
Obesidade,0.817472,0.866749,0.719814,0.675203,0.823171,0.834860,0.762545,0.854560,1.000000



Doença usada como exemplo:
Anemia aplástica

Doenças com sintomas mais semelhantes:


,doenca,similaridade,sintomas
0,"Anemia perniciosa: o que é, sintomas e tratamento",0.8962,os sintomas mais comuns são alteração no funci...
1,Hematúria (presença de sangue na urina),0.8588,"entre todos, o sintoma da hematúria que mais c..."
2,Fratura de costela,0.8440,"dor aguda, intensa e imediata no local do tóra..."
3,Obesidade,0.8175,o principal sintoma da obesidade é o excesso d...
4,"Disfunção erétil: conheça os sintomas, causas ...",0.7768,devido à sua característica principal — não co...



Arquivos finais gerados:
- sintomas_doencas.csv
- doencas_sintomas_embeddings_bert.csv
- matriz_similaridade_doencas_bert.csv
